## Autotuning
Triton has a built-in autotuner that tries different configurations:
```python
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 64}, num_warps=8),
    ],
    key=['M', 'N', 'K'],  # Re-tune when these change
)
@triton.jit
def matmul_kernel(...):
    ...
```

## What the Config Parameters Mean

Each `triton.Config(...)` is one candidate implementation that Triton will benchmark.

```python
triton.Config(
    {'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32},
    num_warps=4,
)
```

- `BLOCK_SIZE_M`: number of rows of `C` computed by one Triton program.
- `BLOCK_SIZE_N`: number of columns of `C` computed by one Triton program.
- `BLOCK_SIZE_K`: size of each chunk along the reduction dimension `K`.
- `num_warps`: number of GPU warps used by one Triton program. More warps can help larger tiles expose more parallelism, but too many can increase overhead or reduce occupancy.
- `num_stages`: number of pipeline stages for loading data and computing. More stages can help hide memory latency, but they may use more registers/shared resources.

The autotune decorator also has important parameters:

- `configs`: all candidate configs Triton should benchmark.
- `key=['M', 'N', 'K']`: retune when these input shape values change.
- `warmup`: how long Triton warms up each candidate before measuring.
- `rep`: how long Triton measures each candidate.
- `cache_results`: whether to cache autotune results across runs when supported.

For matmul, these parameters control the tradeoff between tile reuse, parallelism, register pressure, and occupancy. There is no universally best config, so autotune tries several and keeps the fastest one for the current shape.


## Autotune the matmul kernel

In [8]:
import os

# Ask Triton to print the selected autotune config after benchmarking.
# In a notebook, set this before the first call to the autotuned kernel.
os.environ["TRITON_PRINT_AUTOTUNING"] = "1"

import triton
import triton.language as tl
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE_M': 32,  'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 32}, num_warps=8),
        triton.Config({'BLOCK_SIZE_M': 256, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, num_warps=8),
    ],
    key=['M', 'N', 'K'],  # Re-tune when these change
    warmup=25,
    rep=100,
    cache_results=False,
)
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M,N,K,
    stride_am, stride_ak,
    stride_bk, stride_bn, 
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
   pid_m = tl.program_id(axis=0)
   pid_n = tl.program_id(axis=1)

   offsets_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
   offsets_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
   offsets_k = tl.arange(0, BLOCK_SIZE_K)

   #calculate the ptr of each element for each tiled matrix
   a_ptrs = a_ptr + (offsets_m[:, None]*stride_am + offsets_k[None, :]*stride_ak)
   b_ptrs = b_ptr + (offsets_k[:, None]*stride_bk + offsets_n[None, :]*stride_bn)
   c_ptrs = c_ptr + (offsets_m[:, None]*stride_cm + offsets_n[None, :]*stride_cn)

   accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

   for k in range(0, K, BLOCK_SIZE_K):
      a_mask = (offsets_m[:, None] < M) & (offsets_k[None, :] + k < K)
      b_mask = (offsets_k[:, None] + k < K) & (offsets_n[None, :]< N)

      a = tl.load(a_ptrs+k*stride_ak, mask=a_mask, other=0.0)
      b = tl.load(b_ptrs+k*stride_bk, mask=b_mask, other=0.0)

      accumulator += tl.dot(a, b)

   c_mask = (offsets_m[:, None] < M) & (offsets_n[None, :] < N)
   tl.store(c_ptrs, accumulator, mask=c_mask)
    

In [9]:
import torch
def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_cuda and b.is_cuda
    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float32)

    grid = lambda META: (
        triton.cdiv(M, META['BLOCK_SIZE_M']),
        triton.cdiv(N, META['BLOCK_SIZE_N']),
    )

    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )
    return c

## Run autotune
Run this cell to trigger autotuning. Triton prints the winning configuration after it benchmarks the candidates. If you run the same shape again, Triton may reuse the cached choice; change `M`, `N`, or `K`, or restart the notebook kernel, to force a fresh tuning run.

In [10]:
M, N, K = 1024, 1024, 1024

a = torch.randn((M, K), device="cuda", dtype=torch.float16)
b = torch.randn((K, N), device="cuda", dtype=torch.float16)

# The first call for this shape triggers autotuning.
c = matmul(a, b)
torch.cuda.synchronize()

expected = a @ b
max_diff = torch.max(torch.abs(c - expected)).item()
print(f"max diff: {max_diff}")

# Triton also keeps the selected config on the autotuned kernel object in many versions.
# The environment print above is the most reliable way to see the result.
best_config = getattr(matmul_kernel, "best_config", None)
if best_config is not None:
    print("best_config:", best_config)

Autotuning kernel matmul_kernel with config BLOCK_SIZE_M: 32, BLOCK_SIZE_N: 64, BLOCK_SIZE_K: 32, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel matmul_kernel with config BLOCK_SIZE_M: 64, BLOCK_SIZE_N: 64, BLOCK_SIZE_K: 32, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel matmul_kernel with config BLOCK_SIZE_M: 64, BLOCK_SIZE_N: 128, BLOCK_SIZE_K: 32, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel matmul_kernel with config BLOCK_SIZE_M: 128, BLOCK_SIZE_N: 64, BLOCK_SIZE_K: 32, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel matmul_kernel with config BLOCK_SIZE_M: 128, BLOCK_SIZE_N: 128, BLOCK_SIZE_K: 32, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel matmul_kernel with config BLOCK_SIZE_M: 128, BLOCK_SIZE_N: 256, BLOCK_SIZE_K: 32, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel matmul_kernel with config BLOCK_SIZE_M: 256, BLOCK_S

## Timing benchmark
Run this after the autotune cell. The first `matmul(a, b)` call below reuses the selected config for the same shape, so the timing measures normal kernel execution instead of the tuning search.


In [12]:
import torch.utils.benchmark as benchmark

# Warm up and make sure autotune has already selected a config for this shape.
c = matmul(a, b)
torch.cuda.synchronize()
best_config = getattr(matmul_kernel, "best_config", None)
if best_config is not None:
    print("best_config:", best_config)

triton_timer = benchmark.Timer(
    stmt="matmul(a, b)",
    globals={"matmul": matmul, "a": a, "b": b},
)
torch_timer = benchmark.Timer(
    stmt="a @ b",
    globals={"a": a, "b": b},
)

print("Triton matmul:")
print(triton_timer.timeit(100))

print("PyTorch matmul:")
print(torch_timer.timeit(100))


best_config: BLOCK_SIZE_M: 64, BLOCK_SIZE_N: 64, BLOCK_SIZE_K: 32, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Triton matmul:
matmul(a, b)
  59.37 us
  1 measurement, 100 runs , 1 thread
PyTorch matmul:
a @ b
  20.80 us
  1 measurement, 100 runs , 1 thread
